In [2]:
from torch.utils.data import Dataset
from PIL import Image
import torch
import os

def win_long_path(p):
    """Return a path string safe for Windows' 260-char MAX_PATH limit."""
    p = p.resolve()
    s = str(p)
    if os.name == "nt" and not s.startswith("\\\\?\\"):
        s = "\\\\?\\" + s
    return s

In [3]:
import os
from pathlib import Path

PNG_ROOT = Path("cbis_ddsm_png")

def win_long_path(p: Path) -> str:
    """Return a path string safe for Windows' MAX_PATH limit."""
    p = p.resolve()
    s = str(p)
    if os.name == "nt" and not s.startswith("\\\\?\\"):
        s = "\\\\?\\" + s
    return s

class CBISDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.image_paths = []

        for relative_dir in self.df["image file path"]:
            full_dir = PNG_ROOT / relative_dir
            png_files = sorted(full_dir.rglob("*.png"))

            if len(png_files) == 0:
                raise FileNotFoundError(f"No PNG file found under:\n{full_dir}")

            self.image_paths.append(png_files[0])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image = Image.open(win_long_path(self.image_paths[idx])).convert("RGB")
        label = int(self.df.loc[idx, "pathology"])

        if self.transform:
            image = self.transform(image)

        return image, label

In [4]:
from torchvision import transforms

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [5]:
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

In [5]:
import pandas as pd

train_df = pd.read_csv("../dataframes/full_photo_only_train_df.csv")
test_df = pd.read_csv("../dataframes/full_photo_only_test_df.csv")

train_dataset = CBISDataset(train_df, transform=train_transform)
test_dataset = CBISDataset(test_df, transform=test_transform)

NameError: name 'CBISDataset' is not defined

In [7]:
print(train_df.columns.tolist())
print(train_df.head())

['patient_id', 'pathology', 'image file path']
  patient_id  pathology                                    image file path
0    P_00005          1  Calc-Training_P_00005_RIGHT_CC/1.3.6.1.4.1.959...
1    P_00005          1  Calc-Training_P_00005_RIGHT_MLO/1.3.6.1.4.1.95...
2    P_00007          0  Calc-Training_P_00007_LEFT_CC/1.3.6.1.4.1.9590...
3    P_00007          0  Calc-Training_P_00007_LEFT_MLO/1.3.6.1.4.1.959...
4    P_00008          0  Calc-Training_P_00008_LEFT_CC/1.3.6.1.4.1.9590...


In [8]:
from pathlib import Path

print(Path.cwd())
print((Path("cbis_ddsm_png")).resolve())
print((Path("cbis_ddsm_png")).exists())

c:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment
C:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment\cbis_ddsm_png
True


In [9]:
image, label = train_dataset[0]

print(image.shape)
print(label)

torch.Size([3, 224, 224])
1


## Multi-model training pipeline (ResNet50, DenseNet-121, EfficientNet-B2, VGG16)

Steps:
1. Stratified k-fold CV per model on `train_df` → mean/std metrics per model
2. Retrain each model on the full `train_df`
3. Evaluate each final model once on the held-out `test_df`
4. Save both result tables to CSV

In [10]:
import copy
import numpy as np
import pandas as pd

import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import models

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


In [11]:
# ---- experiment config ----
MODEL_NAMES = ["resnet50", "densenet121", "efficientnet_b2", "vgg16"]

EPOCHS = 10
BATCH_SIZE = 16
LR = 1e-4
RANDOM_STATE = 42
NUM_CLASSES = 2  # binary: pathology 0/1
VAL_SIZE = 0.15  # fraction of train_df held out for monitoring/early-checkpoint selection

# data loading — parallel workers for the DataLoader (see benchmark cell earlier).
# start at 4; if you hit multiprocessing errors in a Windows/Jupyter kernel, drop to 2 or 0.
NUM_WORKERS = 0
PIN_MEMORY = torch.cuda.is_available()

In [12]:
def get_model(name, num_classes=NUM_CLASSES, pretrained=True):
    """Factory that returns one of the 4 backbones with the head swapped
    for `num_classes` outputs, moved to `device`."""
    name = name.lower()

    if name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        model = models.resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif name == "densenet121":
        weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
        model = models.densenet121(weights=weights)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)

    elif name == "efficientnet_b2":
        weights = models.EfficientNet_B2_Weights.DEFAULT if pretrained else None
        model = models.efficientnet_b2(weights=weights)
        in_features = model.classifier[1].in_features
        model.classifier[1] = nn.Linear(in_features, num_classes)

    elif name == "vgg16":
        weights = models.VGG16_Weights.DEFAULT if pretrained else None
        model = models.vgg16(weights=weights)
        in_features = model.classifier[6].in_features
        model.classifier[6] = nn.Linear(in_features, num_classes)

    else:
        raise ValueError(f"Unknown model name: {name}")

    return model.to(device)

In [13]:
def compute_metrics(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    try:
        auc = roc_auc_score(y_true, y_proba)
    except ValueError:
        auc = float("nan")  # happens if a fold/batch has only one class present

    return {"accuracy": acc, "precision": precision, "recall": recall, "f1": f1, "auc": auc}


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_labels, all_preds, all_probs = [], [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)

        probs = torch.softmax(outputs, dim=1)[:, 1]
        preds = torch.argmax(outputs, dim=1)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_probs.extend(probs.cpu().numpy())

    avg_loss = running_loss / len(loader.dataset)
    metrics = compute_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = avg_loss

    return metrics, all_labels, all_preds, all_probs

### Speed fix — pre-resize images to disk once, then load in parallel

The benchmark showed data loading (~230 ms/image) is ~27x slower than GPU compute (~8 ms/image) — every epoch was re-decoding the full-resolution mammogram from scratch just to shrink it to 224x224. This cell resizes every image **once** to a small cache on disk; every fold/epoch after that reads the small cached version instead.

Run this cell once — it skips any file that's already cached, so it's safe to re-run.

In [14]:
from pathlib import Path
from PIL import Image

CACHE_ROOT = Path("cbis_ddsm_cache")
# Slightly larger than the 224 the model trains on, so RandomRotation/RandomCrop-style
# augmentation still has a little margin to work with.
CACHE_SIZE = 256


def build_image_cache(df, source_dataset, split_name):
    """Resize every image in `source_dataset` once and save it to a small flat cache.
    Flat integer filenames (not the original nested UID folder structure) also sidestep
    the Windows MAX_PATH issue entirely for the cache itself."""
    out_dir = CACHE_ROOT / split_name
    out_dir.mkdir(parents=True, exist_ok=True)

    n = len(df)
    for idx in range(n):
        out_path = out_dir / f"{idx}.png"
        if out_path.exists():
            continue  # already cached

        src_path = win_long_path(source_dataset.image_paths[idx])
        image = Image.open(src_path).convert("RGB")
        image = image.resize((CACHE_SIZE, CACHE_SIZE), Image.BILINEAR)
        image.save(out_path)

        if (idx + 1) % 200 == 0 or (idx + 1) == n:
            print(f"[{split_name}] cached {idx + 1}/{n}")


# transform=None here: we only need these to resolve the original (slow) file paths,
# no image loading/transforming happens until build_image_cache() actually opens each file.
_train_source = CBISDataset(train_df, transform=None)
_test_source = CBISDataset(test_df, transform=None)

build_image_cache(train_df, _train_source, "train")
build_image_cache(test_df, _test_source, "test")

print("\nCaching done.")


Caching done.


In [15]:
class CBISCachedDataset(Dataset):
    """Reads pre-resized images from the on-disk cache built by build_image_cache().
    Far faster than CBISDataset since it never touches the original full-resolution file."""

    def __init__(self, dataframe, split_name, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.cache_dir = CACHE_ROOT / split_name
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image = Image.open(self.cache_dir / f"{idx}.png").convert("RGB")
        label = int(self.df.loc[idx, "pathology"])

        if self.transform:
            image = self.transform(image)

        return image, label


# Rebind the dataset objects the rest of the pipeline uses to the fast, cached versions.
train_dataset_aug = CBISCachedDataset(train_df, "train", transform=train_transform)
train_dataset_eval = CBISCachedDataset(train_df, "train", transform=test_transform)
test_dataset = CBISCachedDataset(test_df, "test", transform=test_transform)

### Stage 1 — Single stratified train/validation split

**Note on methodology:** this uses one stratified train/val split per model rather than k-fold CV, to keep total training time practical on local hardware. The validation split is used only to monitor for overfitting and pick the best-performing epoch's weights — the number that should be reported as the result is the test-set evaluation in Stage 2, which each model only sees once. This is a reasonable, defensible simplification for a thesis, but it's worth stating explicitly as a limitation: single-split metrics carry more variance than k-fold means, so a difference between two models here could partly reflect how the split happened to fall rather than a true difference in ability.

In [16]:
from sklearn.model_selection import train_test_split

train_idx, val_idx = train_test_split(
    np.arange(len(train_df)),
    test_size=VAL_SIZE,
    stratify=train_df["pathology"].values,
    random_state=RANDOM_STATE,
)

print(f"train split: {len(train_idx)} images | val split: {len(val_idx)} images")

train_split_subset = Subset(train_dataset_aug, train_idx)
val_split_subset = Subset(train_dataset_eval, val_idx)

train_loader = DataLoader(train_split_subset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                           persistent_workers=(NUM_WORKERS > 0))
val_loader = DataLoader(val_split_subset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                         persistent_workers=(NUM_WORKERS > 0))

train split: 2159 images | val split: 382 images


In [17]:
def train_model_single_split(model_name, epochs=EPOCHS, lr=LR):
    model = get_model(model_name)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    best_val_f1 = -1.0
    best_state = None
    history = []

    for epoch in range(1, epochs + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        val_metrics, _, _, _ = evaluate(model, val_loader, criterion)
        history.append({"epoch": epoch, "train_loss": train_loss, **val_metrics})

        print(f"[{model_name}] epoch {epoch}/{epochs} | train_loss {train_loss:.4f} "
              f"| val_loss {val_metrics['loss']:.4f} | val_acc {val_metrics['accuracy']:.4f} "
              f"| val_f1 {val_metrics['f1']:.4f} | val_auc {val_metrics['auc']:.4f}")

        # keep the best epoch's weights (by val F1) as protection against overfitting
        # in later epochs — this is a lightweight stand-in for early stopping
        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_state = copy.deepcopy(model.state_dict())

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, pd.DataFrame(history)

### Stage 2 — Evaluate once on the held-out test set

Each model is evaluated on `test_df` exactly once, using its best-val-F1 checkpoint from Stage 1. These are the numbers to report as your final results.

In [18]:
import os
cache_dir = os.path.expanduser("~/.cache/torch/hub/checkpoints")
if os.path.exists(cache_dir):
    print(os.listdir(cache_dir))
else:
    print("cache dir doesn't exist yet — nothing downloaded")

['densenet121-a639ec97.pth', 'efficientnet_b2_rwightman-c35c1473.pth', 'resnet18-f37072fd.pth', 'resnet50-0676ba61.pth', 'resnet50-11ad3fa6.pth', 'vgg16-397923af.pth']


In [19]:
val_histories = {}
final_results = {}
trained_models = {}

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                          persistent_workers=(NUM_WORKERS > 0))
criterion = nn.CrossEntropyLoss()

for model_name in MODEL_NAMES:
    print(f"\n===== {model_name}: train + validate =====")
    model, history = train_model_single_split(model_name)
    trained_models[model_name] = model
    val_histories[model_name] = history

    test_metrics, y_true, y_pred, y_proba = evaluate(model, test_loader, criterion)
    print(f"[{model_name}] TEST metrics: {test_metrics}")
    final_results[model_name] = test_metrics

    torch.save(model.state_dict(), f"{model_name}_final.pt")

test_results_df = pd.DataFrame(final_results).T
test_results_df


===== resnet50: train + validate =====
[resnet50] epoch 1/10 | train_loss 0.6391 | val_loss 0.6814 | val_acc 0.5733 | val_f1 0.6183 | val_auc 0.6641
[resnet50] epoch 2/10 | train_loss 0.5879 | val_loss 0.5817 | val_acc 0.7042 | val_f1 0.6502 | val_auc 0.7617
[resnet50] epoch 3/10 | train_loss 0.5448 | val_loss 0.5513 | val_acc 0.7120 | val_f1 0.6358 | val_auc 0.7784
[resnet50] epoch 4/10 | train_loss 0.5027 | val_loss 0.5760 | val_acc 0.6937 | val_f1 0.5145 | val_auc 0.7573
[resnet50] epoch 5/10 | train_loss 0.4778 | val_loss 0.5572 | val_acc 0.7147 | val_f1 0.6007 | val_auc 0.7802
[resnet50] epoch 6/10 | train_loss 0.4321 | val_loss 0.4994 | val_acc 0.7408 | val_f1 0.6400 | val_auc 0.8305
[resnet50] epoch 7/10 | train_loss 0.4000 | val_loss 0.5483 | val_acc 0.7147 | val_f1 0.5477 | val_auc 0.8240
[resnet50] epoch 8/10 | train_loss 0.3698 | val_loss 0.5205 | val_acc 0.7277 | val_f1 0.6061 | val_auc 0.8376
[resnet50] epoch 9/10 | train_loss 0.3270 | val_loss 0.6033 | val_acc 0.7120 | v

,accuracy,precision,recall,f1,auc,loss
resnet50,0.695853,0.635088,0.658182,0.646429,0.741721,0.643601
densenet121,0.675883,0.636752,0.541818,0.585462,0.758685,0.590483
efficientnet_b2,0.717358,0.682731,0.618182,0.648855,0.779052,0.698728
vgg16,0.645161,0.550000,0.880000,0.676923,0.733201,0.602160


### Save results

In [20]:
test_results_df.to_csv("test_results.csv")

for model_name, history in val_histories.items():
    history.to_csv(f"{model_name}_val_history.csv", index=False)

print("Saved test_results.csv and per-model val history CSVs "
      "(useful for plotting train/val loss curves per model in your write-up).")

Saved test_results.csv and per-model val history CSVs (useful for plotting train/val loss curves per model in your write-up).
